[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mongodb-partners/partner-cookbooks/blob/main/advanced-rag/01_atlas_setup_and_vector_search.ipynb)

# Advanced RAG on MongoDB Atlas: 1. Setup & Vector Search

This is the first of three **Advanced RAG** notebooks. Together they
build a movie-discovery assistant over the `sample_mflix` dataset, one retrieval technique at a
time. By the end of this notebook you will have a working semantic-search retriever and a first
retrieval-augmented generation (RAG) chain answering questions about movies.

## What you'll learn
1. Connect to a free MongoDB Atlas cluster and inspect the `sample_mflix` sample data.
2. Understand the plot embeddings that ship with the dataset.
3. Create an Atlas **Vector Search** index from Python.
4. Run semantic search with `MongoDBAtlasVectorSearch`, including metadata pre-filtering.
5. Wire the retriever into a RAG chain with Claude.

## Prerequisites
- **A free Atlas cluster (M0).** Create one at [mongodb.com/atlas](https://www.mongodb.com/atlas)
  and [load the sample dataset](https://www.mongodb.com/docs/atlas/sample-data/). It includes
  `sample_mflix` with pre-computed plot embeddings.
- **An Atlas API key** for the hosted embedding service, and a **Claude API key**.
- Beginner Python. No prior vector-search experience required.


## Setup

Install the libraries. On Colab this runs on a fresh machine; locally it's a no-op if you already installed `requirements.txt`.

In [1]:
%%capture
%pip install -q \
    langchain==1.3.12 langchain-mongodb==0.11.0 langchain-anthropic==1.4.8 \
    pymongo==4.17.0 python-dotenv==1.2.2 requests==2.34.2

### Credentials

The notebooks read settings from environment variables (via a local `.env` file if present) and
fall back to a secure prompt when a value is missing, so this works the same on Colab or your
laptop. See `.env.example` for the full list.


In [2]:
import os
from dotenv import load_dotenv

# `override=True` lets a local .env win over any pre-set shell variables.
load_dotenv(override=True)

from utils import get_secret  # shared helpers live in advanced-rag/utils.py

MONGODB_URI  = get_secret("MONGODB_URI",  "MongoDB Atlas connection string: ")
ATLAS_AI_KEY = get_secret("MDB_ATLAS_API_KEY", "Atlas API key (for embeddings): ")

EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL", "voyage-3-large")
EMBEDDING_DIMS  = int(os.getenv("EMBEDDING_DIMENSIONS", "2048"))
DB_NAME         = os.getenv("MONGODB_DB", "sample_mflix")
print("Configured for model", EMBEDDING_MODEL, f"({EMBEDDING_DIMS}-d)")

Configured for model voyage-3-large (2048-d)


## 1. Connect to Atlas and look at the data

`utils.connect` opens a client and pings the server so a bad connection string fails immediately.

In [3]:
from utils import connect

client = connect(MONGODB_URI)
db = client[DB_NAME]
movies = db["embedded_movies"]

print("Collections:", db.list_collection_names())
print("embedded_movies documents:", movies.estimated_document_count())

Collections: ['movies', 'users', 'embedded_movies', 'sessions', 'comments', 'theaters']
embedded_movies documents: 3483


Each movie carries its title, plot, genres, cast, directors, and, importantly, **pre-computed
plot embeddings**. The `sample_mflix.embedded_movies` collection ships with two vector fields;
we use `plot_embedding_voyage_3_large`, a 2048-dimensional embedding produced by Voyage AI's
`voyage-3-large` model. Because the vectors are already there, we can jump straight to indexing.


In [4]:
doc = movies.find_one(
    {"plot_embedding_voyage_3_large": {"$exists": True}},
    {"title": 1, "plot": 1, "genres": 1, "cast": 1, "plot_embedding_voyage_3_large": 1},
)
print("Title :", doc["title"])
print("Genres:", doc["genres"])
print("Plot  :", doc["plot"][:160], "...")

vec = doc["plot_embedding_voyage_3_large"].as_vector()  # BSON binary -> float vector
print(f"\nStored embedding: {len(vec.data)} dimensions, dtype {vec.dtype}")

Title : Scarface
Genres: ['Action', 'Crime', 'Drama']
Plot  : An ambitious and near insanely violent gangster climbs the ladder of success in the mob, but his weaknesses prove to be his downfall. ...

Stored embedding: 2048 dimensions, dtype BinaryVectorDtype.FLOAT32


### Why the query model must match

Vector search compares a **query vector** against the **stored vectors**. They only line up if
both come from the *same* embedding model. The stored plots use `voyage-3-large` at 2048
dimensions, so our query embedder must too. `AtlasEmbeddings` (in `utils.py`) calls MongoDB's
hosted embedding service and validates the dimension on every call, so a mismatch fails loudly
instead of silently returning nonsense.


In [5]:
from utils import AtlasEmbeddings

embeddings = AtlasEmbeddings(
    api_key=ATLAS_AI_KEY, model=EMBEDDING_MODEL, dimensions=EMBEDDING_DIMS,
)

# Sanity check: a query embedding has the same width as the stored vectors.
q = embeddings.embed_query("a thrilling space adventure")
assert len(q) == EMBEDDING_DIMS, "query dim must match stored dim"
print("Query embedding dimensions:", len(q))

Query embedding dimensions: 2048


## 2. Create a Vector Search index

Atlas needs a **vector search index** to run semantic queries efficiently. We create it from
Python and declare two `filter` fields (`genres`, `year`) so we can combine semantic search with
metadata filtering later. `wait_until_complete` blocks until the index is queryable; on M0 this
usually takes a minute or two. Re-running is safe: we skip creation if the index already exists.


In [6]:
from langchain_mongodb.index import create_vector_search_index

VECTOR_INDEX = "movies_vector_index"

if VECTOR_INDEX not in {ix["name"] for ix in movies.list_search_indexes()}:
    print("Creating vector index, waiting until it is queryable...")
    create_vector_search_index(
        collection=movies,
        index_name=VECTOR_INDEX,
        path="plot_embedding_voyage_3_large",
        dimensions=EMBEDDING_DIMS,
        similarity="cosine",
        filters=["genres", "year"],
        wait_until_complete=420,
    )
    print("Index is ready.")
else:
    print("Index already exists.")

Index already exists.


## 3. Semantic search with `MongoDBAtlasVectorSearch`

`MongoDBAtlasVectorSearch` is LangChain's wrapper around Atlas Vector Search. We point it at the
collection, the embedding model, and the fields holding the plot text and its vector. Then
`similarity_search` embeds the query and returns the closest movies by meaning, not keywords.


In [7]:
from langchain_mongodb import MongoDBAtlasVectorSearch

vector_store = MongoDBAtlasVectorSearch(
    collection=movies,
    embedding=embeddings,
    index_name=VECTOR_INDEX,
    text_key="plot",
    embedding_key="plot_embedding_voyage_3_large",
    relevance_score_fn="cosine",
)

results = vector_store.similarity_search("a criminal mastermind planning a heist", k=5)
for r in results:
    print(f"- {r.metadata['title']:<35} {r.metadata.get('genres')}")

- Parker                              ['Action', 'Crime', 'Thriller']
- The Hot Rock                        ['Action', 'Comedy', 'Crime']
- The Italian Job                     ['Action', 'Crime', 'Thriller']
- After the Sunset                    ['Action', 'Comedy', 'Crime']
- Tower Heist                         ['Action', 'Comedy', 'Crime']


Notice the results are thematically related to *heist* even when the plots never use that word.
That is semantic search at work.

### Pre-filtering: search within a subset

Because we declared `genres` as a filter field, we can restrict the search to a genre and still
rank by meaning. This runs the filter and the vector search together inside Atlas.


In [8]:
fantasy = vector_store.similarity_search(
    "an epic quest to save the kingdom", k=5,
    pre_filter={"genres": {"$eq": "Fantasy"}},
)
for r in fantasy:
    print(f"- {r.metadata['title']:<35} {r.metadata.get('genres')}")

- The Sword and the Sorcerer          ['Action', 'Fantasy', 'Adventure']
- Krull                               ['Action', 'Adventure', 'Fantasy']
- Your Highness                       ['Adventure', 'Comedy', 'Fantasy']
- Mythica: A Quest for Heroes         ['Action', 'Adventure', 'Fantasy']
- Ubit drakona                        ['Drama', 'Family', 'Fantasy']


## 4. A first RAG chain

Retrieval-augmented generation = *retrieve* relevant context, then let the model *generate* an
answer grounded in it. We turn the vector store into a retriever, format the retrieved plots as
context, and hand it to Claude with a prompt that says: answer only from the context.

`build_llm_kwargs` configures `ChatAnthropic` for either the Anthropic API or a compatible
gateway, depending on whether `ANTHROPIC_BASE_URL` is set.


In [9]:
from langchain_anthropic import ChatAnthropic
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from utils import build_llm_kwargs

ANTHROPIC_KEY = get_secret("ANTHROPIC_API_KEY", "Claude API key: ")
llm = ChatAnthropic(**build_llm_kwargs(
    api_key=ANTHROPIC_KEY,
    model=os.getenv("ANTHROPIC_MODEL", "claude-haiku-4-5"),
    base_url=os.getenv("ANTHROPIC_BASE_URL") or None,
    max_tokens=512,
))

retriever = vector_store.as_retriever(search_kwargs={"k": 5})

def format_docs(docs):
    return "\n\n".join(f"{d.metadata['title']}: {d.page_content}" for d in docs)

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a movie expert. Recommend movies using ONLY the context. "
               "For each, give the title and one sentence on why it fits."),
    ("human", "Context:\n{context}\n\nQuestion: {question}"),
])

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt | llm | StrOutputParser()
)

print(rag_chain.invoke("I want a tense movie about a robbery or heist. Any suggestions?"))

# Heist Movie Recommendations

**Heat** - This film delivers intense tension as professional bank robbers face mounting pressure from police after leaving crucial evidence at a heist scene.

**Kaante** - The cat-and-mouse dynamics intensify dramatically when the robbery crew discovers one member is actually an undercover cop working against them.

**Takers** - This movie builds tension through the clash between ambitious bank robbers executing a major heist and a relentless detective determined to stop them.


## Recap

You connected to Atlas, used the plot embeddings that ship with `sample_mflix`, built a vector
search index, ran semantic search with and without metadata filters, and grounded a Claude answer
in the results.

**Next:** `02_fulltext_hybrid_and_reranking.ipynb` adds full-text (keyword) search, combines it
with vector search using MongoDB's native `$rankFusion` **hybrid** retrieval, and improves ranking
with a **reranker**.
